# Aion 1.0 Instruct 2.3B — on-device SLM on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Aion-1.0-Instruct-2.3B-on-device-slm.ipynb)

Run Microsoft's [`microsoft/Aion-1.0-Instruct-2.3B`](https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B) locally: a **2.3B MIT instruct SLM** from the Windows Applied Sciences Group, tagged for **summarization, rewriting, and chat**. The card lists eight languages (English, German, Spanish, Portuguese, Italian, French, Chinese, Japanese), a **32,768-token** context, and custom modeling code (`trust_remote_code=True`). Knowledge cutoff is **2025-12**. A further-optimized Windows OS SLM is planned by the end of January 2027; this Hub dump is the unquantized introduction.

See [use cases](https://github.com/donvito/notebooks/blob/main/docs/Aion-1.0-Instruct-2.3B-on-device-slm.md) for when this notebook is a good fit.

**Why this model.** It is small enough for a free T4, licensed MIT, and the card's intended jobs are exactly the three demos below: summarize a long email thread, rewrite a paragraph, and chat in a non-English language. The bundled template is **not ChatML**. Every turn is `<|system|>` / `<|user|>` / `<|assistant|>` closed by `<|end|>` (token **200020**, also EOS).

**Do not copy the card's `dtype="auto"` on a T4.** `config.json` stores `"dtype": "bfloat16"`. T4 (`sm_75`) has **no native bfloat16** and **no FlashAttention-2**. This notebook loads **float16** (~4.6 GB weights) with **SDPA** (eager fallback), then **float32** (~9.2 GB, still fits 15 GB) if a short probe is NaN or gibberish.

**Pin transformers 5.9.0.** The checkpoint was exported with `transformers_version` **5.9.0**. A [public write-up](https://x.com/prompt48/status/2124597425310695696) reports that the card quickstart on **5.24.0** returned gibberish with the KV cache on (`BANer h2 circles $ scarce` for a one-word prompt) and decoded cleanly on **5.9.0**. This notebook pins 5.9.0 and does **not** upgrade to Colab's 5.24.x.

**Generation.** The Hub snippet is greedy (`model.generate(..., max_new_tokens=512)`). `generation_config.json` sets `eos_token_id` to `[200020, 199999]`, `pad_token_id` 199999, `use_cache: true`, and no temperature. The Windows sample [`AionInstructClient.cs`](https://github.com/microsoft/Aion-Instruct-Preview-Sample/blob/main/AionInstructClient.cs) uses `Temperature = 0.5`, `TopP = 0.9`, `TopK = 40` for chat. Summarize / rewrite here stay **greedy**; the multilingual chat turn uses those sample settings.

**Licence:** MIT. Weights are public — cancel any Colab `HF_TOKEN` prompt.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. Expected peak VRAM is about **5–7 GB** in float16, or about **10 GB** if the notebook falls back to float32. Keep prompts short; a T4 cannot hold a full 32K KV cache on top of the weights.

**Links:** [model card](https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B) · [Windows sample](https://github.com/microsoft/Aion-Instruct-Preview-Sample) · [announcement](https://x.com/mswnlz/status/2123114277330395257) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Aion-1.0-Instruct-2.3B-on-device-slm.md)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so the later transformers pin can take effect. T4 (`sm_75`) has no native bfloat16 — **float16**, then **float32** if needed, is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Never pick bfloat16 from dtype="auto" on this GPU. Weights on the Hub are bf16;
# we cast to float16 (or float32 later) instead of emulating bf16 on sm_75.
DTYPE = torch.float16
MODEL_ID = "microsoft/Aion-1.0-Instruct-2.3B"

free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB")
print("dtype:", DTYPE, "| native bf16:", NATIVE_BF16)
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("T4 path: float16 first, float32 if the probe is NaN/gibberish. bfloat16 is off.")


## 2. Install the pinned transformers

The [model card](https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B) requires custom code (`trust_remote_code=True`) and was exported with **transformers 5.x**. `config.json` and `generation_config.json` record **5.9.0**. Colab often ships **5.24.x**, which is the version a public report found produced gibberish with the KV cache on.

This cell pins **`transformers==5.9.0`**, then drops any already-imported `transformers` modules so **Runtime > Run all** can pick it up without *Restart session*. If the version assert still fails, restart once and continue from the GPU cell.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`.


In [ ]:
%pip install -q "transformers==5.9.0" "accelerate>=1.1.0" huggingface_hub

import sys

# Drop a pre-imported Colab transformers so 5.9.0 is visible without Restart session.
for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) == Version("5.9.0"), (
    f"transformers {transformers.__version__} is loaded; Aion 1.0 Instruct was exported with 5.9.0. "
    "Runtime > Restart session, then rerun from the GPU check cell."
)
print("transformers", transformers.__version__)


> If the version assert failed, use *Runtime > Restart session* and continue from the GPU-check cell (the pin is already installed).


## 3. Load Aion 1.0 Instruct 2.3B

`AutoTokenizer` + `AutoModelForCausalLM` with `trust_remote_code=True`, as on the [card](https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B#quickstart). First run downloads **`model.safetensors` (~4.6 GB, bf16 on disk)**. Attention is **sdpa**, then **eager**. Dtype is **float16**, then **float32** if a one-word probe is NaN or non-text.

The tokenizer ships `chat_template.jinja`. Do not hand-write `<|user|>` tags — the [Windows prompt guide](https://github.com/microsoft/Aion-Instruct-Preview-Sample/blob/main/docs/prompt-guide.md) says to use the model's own template.


In [ ]:
from datetime import date

from transformers import AutoModelForCausalLM, AutoTokenizer

CURRENT_DATE = date.today().isoformat()
print("chat-template current_date:", CURRENT_DATE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print("eos_token:", repr(tokenizer.eos_token), "| pad_token:", repr(tokenizer.pad_token))
print("bos_token:", repr(tokenizer.bos_token))

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_causal(dtype, attn):
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=dtype,
        attn_implementation=attn,
        device_map="auto",
        trust_remote_code=True,
    ).eval()


def probe_ok(text, finite):
    # Card-style one-word probe. A public 5.24.0 + cache report decoded
    # "BANer h2 circles $ scarce" here; 5.9.0 decoded banana.
    return bool(finite) and "banana" in (text or "").lower()


def probe_reply(model, use_cache=True):
    messages = [{"role": "user", "content": "Reply with the single word banana."}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        current_date=CURRENT_DATE,
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            use_cache=use_cache,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=model.generation_config.pad_token_id,
        )
    text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    finite = True
    with torch.inference_mode():
        logits = model(**inputs).logits
        finite = bool(torch.isfinite(logits).all().item())
    return text, finite


ATTN = None
DTYPE_IN_USE = None
USE_CACHE = True
model = None
last_error = None
probe_text = None

for dtype in (torch.float16, torch.float32):
    for attn in ("sdpa", "eager"):
        try:
            if model is not None:
                del model
                torch.cuda.empty_cache()
            print(f"loading dtype={dtype} attn={attn} ...")
            model = load_causal(dtype, attn)
            ATTN = attn
            DTYPE_IN_USE = dtype
            probe_text, finite = probe_reply(model, use_cache=True)
            print(f"probe (use_cache=True): {probe_text!r} | finite logits: {finite}")
            if probe_ok(probe_text, finite):
                break
            print("probe looked broken; trying the next dtype/attn.")
        except Exception as exc:
            last_error = exc
            print(f"{attn} {dtype} load/probe failed:", exc)
            del model
            model = None
            torch.cuda.empty_cache()
    else:
        continue
    break

if model is None:
    raise RuntimeError("could not load microsoft/Aion-1.0-Instruct-2.3B") from last_error

if not probe_ok(probe_text, True):
    print("retrying probe with use_cache=False (workaround if a newer transformers slipped in)")
    probe_text, finite = probe_reply(model, use_cache=False)
    print(f"probe (use_cache=False): {probe_text!r} | finite logits: {finite}")
    if probe_ok(probe_text, finite):
        USE_CACHE = False

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("attention:", ATTN, "| dtype:", DTYPE_IN_USE, "| use_cache:", USE_CACHE)
print("generation_config:", model.generation_config)
print(f"parameters: {n_params:.2f}B")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)
if not probe_ok(probe_text, True):
    print("WARNING: the banana probe still looks wrong. Check the transformers pin and chat template.")


## 4. Chat-template helper

`apply_chat_template(..., add_generation_prompt=True)` prepends the card's system preamble (`You are Aion Instruct...`, knowledge cutoff `2025-12`, current date) and ends the prompt on `<|assistant|>`. The first call prints the rendered prompt so you can confirm it is **not** ChatML.

Greedy is the Hub default. Pass `sample=True` for the Windows sample's `0.5 / 0.9 / 40` chat settings.


In [ ]:
SHOWED_PROMPT = False


def generate(messages, max_new_tokens=256, sample=False, show_prompt=False):
    global SHOWED_PROMPT
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        current_date=CURRENT_DATE,
    ).to(model.device)
    if show_prompt or not SHOWED_PROMPT:
        print("--- rendered prompt (special tokens visible) ---")
        print(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False))
        print("--- end prompt ---")
        SHOWED_PROMPT = True
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        use_cache=USE_CACHE,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=model.generation_config.pad_token_id,
    )
    if sample:
        # LanguageModelOptions in AionInstructClient.cs
        gen_kwargs.update(do_sample=True, temperature=0.5, top_p=0.9, top_k=40)
    else:
        gen_kwargs.update(do_sample=False)
    with torch.inference_mode():
        output = model.generate(**inputs, **gen_kwargs)
    text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    return text.strip()


## 5. Demo 1 — summarize a short email thread

The card's uses include document comprehension and lengthy email threads on-device. The [prompt guide](https://github.com/microsoft/Aion-Instruct-Preview-Sample/blob/main/docs/prompt-guide.md) says to quote user-provided text as **data** and put the output instruction last. Greedy decoding, matching the Hub snippet.


In [ ]:
EMAIL_THREAD = """From: Priya Chen <priya@example.com>
To: Marco Rossi <marco@example.com>
Sent: Monday 09:12
Subject: Q2 launch one-pager

Need the Q2 launch one-pager by Thursday. Legal still has comments on the pricing slide, so we cannot lock numbers yet. Please keep the rest of the deck moving.

From: Marco Rossi <marco@example.com>
To: Priya Chen <priya@example.com>
Sent: Monday 11:40

I'll send a draft tonight. Can we ship Thursday without the pricing numbers and drop them in on Friday after Legal signs off? Jordan should also see the EU copy before we send it out.

From: Priya Chen <priya@example.com>
To: Marco Rossi <marco@example.com>
Cc: Jordan Blake <jordan@example.com>
Sent: Monday 12:05

Yes — ship the rest Thursday. Flag the pricing page as TBD. Jordan, please review the EU copy tomorrow morning so Marco can merge comments before Thursday.
"""

messages = [
    {
        "role": "user",
        "content": (
            "Email thread:\n"
            f'"""{EMAIL_THREAD}"""\n\n'
            "Summarize this thread in 4–6 bullet points. Cover the decision, the Thursday "
            "deadline, what is still TBD, and who owes what. Do not invent facts. "
            "Respond with bullets only."
        ),
    }
]
print(generate(messages, max_new_tokens=256, sample=False, show_prompt=True))


## 6. Demo 2 — rewrite a paragraph in a different tone

Rewriting is a card tag and a first-class tab in the [Windows sample](https://github.com/microsoft/Aion-Instruct-Preview-Sample) (`TextRewriter`). Here the instruction is a tone change, still through the instruct chat template (the Windows Text Intelligence APIs are NPU-only and are not used in Colab).


In [ ]:
STIFF = (
    "Please be advised that the Q2 launch one-pager must be circulated to all "
    "stakeholders no later than Thursday. Pricing figures remain subject to Legal "
    "review and shall be designated TBD until such review is complete. Kindly ensure "
    "that the EU copy is submitted to Jordan Blake for examination prior to distribution."
)

messages = [
    {
        "role": "user",
        "content": (
            "Paragraph:\n"
            f'"""{STIFF}"""\n\n'
            "Rewrite this as a short, friendly Slack message from Priya to Marco. "
            "Keep every fact. Do not add a greeting essay. Output only the rewritten message."
        ),
    }
]
print(generate(messages, max_new_tokens=256, sample=False))


## 7. Demo 3 — a short Spanish chat turn

The card lists Spanish among the eight languages and notes that output quality may differ by language. This is one user turn in Spanish, sampled with the Windows sample's `0.5 / 0.9 / 40` settings. The system preamble still comes from the model's template.


In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "En dos frases, ¿por qué conviene resumir un hilo de correo en el dispositivo "
            "y no enviarlo a un servicio en la nube? Responde en español."
        ),
    }
]
print(generate(messages, max_new_tokens=192, sample=True))


## 8. Peak VRAM

Allocated / reserved / peak after the demos. Float16 weights are ~4.6 GB; float32 is ~9.2 GB. The KV cache for these short turns is small compared with a full 32K window.


In [ ]:
print(f"dtype in use: {DTYPE_IN_USE} | attention: {ATTN} | use_cache: {USE_CACHE}")
print(f"parameters: {n_params:.2f}B")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
free_now, total_now = torch.cuda.mem_get_info()
print(f"free / total: {free_now / 1024**3:.2f} / {total_now / 1024**3:.2f} GB")


## 9. Cleanup

- **Chat template.** Use `tokenizer.apply_chat_template`. Hand-rolled ChatML or missing `add_generation_prompt=True` is out of distribution. `<|end|>` is EOS.
- **Transformers pin.** Stay on **5.9.0**. Newer 5.24.x plus `use_cache=True` is the combination a public report found produced gibberish on the card quickstart. If you must use a newer transformers, try `use_cache=False` first.
- **Dtype.** Hub weights are bf16. A T4 has no native bf16, so this notebook uses float16 and falls back to float32. Do not pass `dtype="auto"` on `sm_75`.
- **No FlashAttention-2** on T4. SDPA / eager only.
- **32K context** is a card limit, not a T4 demo. Keep prompts short.
- **World knowledge.** The card: the model was not trained to excel at it. Prefer jobs over text you supply (summarize, rewrite, chat).
- **Out of scope (card).** Harmful or misleading content; automated decisions in medical or legal contexts.
- **Windows NPU sample** ([microsoft/Aion-Instruct-Preview-Sample](https://github.com/microsoft/Aion-Instruct-Preview-Sample)) is a Copilot+ PC WinUI app, not this Transformers path.
- **Quantized builds.** The card says quantized versions will follow. An [ONNX WebGPU INT4](https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B-ONNX-WebGPU) sibling exists; this notebook loads the bf16 safetensors.
- **HF token.** Public MIT weights. This notebook sets `HF_HUB_DISABLE_IMPLICIT_TOKEN=1`.
- **Tool calling.** The template supports `<|tool_call|>` JSON; it is not one of the demos here.

The next cell drops the model so you can load something else on the same T4.


In [ ]:
import gc

print(
    f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"after cleanup free / total: {free_b / 1024**3:.2f} / {total_b / 1024**3:.1f} GB")
print("Aion 1.0 Instruct 2.3B released. Re-run the load cell if you want another prompt.")
